# DPPH 30 min Molecular Embedding Regression Benchmark

Objective: compare ECFP, RDKit descriptors, and three chemical transformer representations with four regression models for predicting `minusLogIC50 (M)` from pre-split `train.csv`, `val.csv`, and `test.csv`.

Representations:
- ECFP/Morgan fingerprints
- RDKit molecular descriptors
- DeepChem/ChemBERTa-77M-MTR transformer embeddings
- DeepChem/ChemBERTa-77M-MLM transformer embeddings
- DeepChem/MoLFormer-c3-1.1B transformer embeddings

Models:
- Random Forest
- Support Vector Regression
- Lasso
- XGBoost

Protocol: read fixed train/validation/test splits, run 5-fold hyperparameter optimization only on the training split, then evaluate the refit model on validation and test splits.


## 1. Setup

Run the cells top-to-bottom. The notebook assumes `train.csv`, `val.csv`, and `test.csv` are in the same folder as this notebook.


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator

from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
TRAIN_PATH = Path("train.csv")
VAL_PATH = Path("val.csv")
TEST_PATH = Path("test.csv")

TARGET_COLUMN = "minusLogIC50 (M)"
SMILES_COLUMN = "Canonical SMILES"
CV_FOLDS = 5

# ECFP settings
ECFP_RADIUS = 2
ECFP_N_BITS = 2048

# Transformer embedding settings, aligned with the DeepChem Hugging Face pages.
# ChemBERTa-77M-MTR: AutoTokenizer + RobertaForRegression
# ChemBERTa-77M-MLM: AutoTokenizer + AutoModelForMaskedLM
# MoLFormer-c3-1.1B: AutoTokenizer + AutoModelForMaskedLM + trust_remote_code=True
TRANSFORMER_MODELS = {
    # "ChemBERTa_77M_MTR": {
    #     "model_name": "DeepChem/ChemBERTa-77M-MTR",
    #     "model_class": "roberta_regression",
    #     "trust_remote_code": False,
    # },
    "ChemBERTa_77M_MLM": {
        "model_name": "DeepChem/ChemBERTa-77M-MLM",
        "model_class": "masked_lm",
        "trust_remote_code": False,
    },
    # "MoLFormer_c3_1.1B": {
    #     "model_name": "DeepChem/MoLFormer-c3-1.1B",
    #     "model_class": "masked_lm",
    #     "trust_remote_code": True,
    # },
}
TRANSFORMER_BATCH_SIZE = 32
TRANSFORMER_MAX_LENGTH = 256

print(f"Train path: {TRAIN_PATH.resolve()}")
print(f"Val path: {VAL_PATH.resolve()}")
print(f"Test path: {TEST_PATH.resolve()}")


## 2. Load Fixed Train/Val/Test Splits


In [ ]:
# No Excel dependency is required here because this notebook reads the pre-split CSV files.
for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing split file: {path.resolve()}")


In [ ]:
def read_split(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


train_raw = read_split(TRAIN_PATH)
val_raw = read_split(VAL_PATH)
test_raw = read_split(TEST_PATH)

required = [SMILES_COLUMN, TARGET_COLUMN]
for split_name, split_df in [("train", train_raw), ("val", val_raw), ("test", test_raw)]:
    missing = [col for col in required if col not in split_df.columns]
    if missing:
        raise ValueError(
            f"{split_name} is missing required columns: {missing}. "
            f"Available columns: {split_df.columns.tolist()}"
        )

print(f"Raw train shape: {train_raw.shape}")
print(f"Raw val shape: {val_raw.shape}")
print(f"Raw test shape: {test_raw.shape}")
train_raw.head()


In [ ]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol


def clean_split(split_df, split_name):
    cleaned = split_df[required].dropna().copy()
    cleaned[TARGET_COLUMN] = pd.to_numeric(cleaned[TARGET_COLUMN], errors="coerce")
    cleaned = cleaned.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
    cleaned["mol"] = cleaned[SMILES_COLUMN].apply(smiles_to_mol)

    invalid_count = cleaned["mol"].isna().sum()
    if invalid_count:
        print(f"Dropping {invalid_count} invalid SMILES rows from {split_name}")
        cleaned = cleaned.dropna(subset=["mol"]).reset_index(drop=True)

    return cleaned


train_df = clean_split(train_raw, "train")
val_df = clean_split(val_raw, "val")
test_df = clean_split(test_raw, "test")

X_train_smiles = train_df[SMILES_COLUMN].to_numpy()
y_train = train_df[TARGET_COLUMN].to_numpy(dtype=float)

X_val_smiles = val_df[SMILES_COLUMN].to_numpy()
y_val = val_df[TARGET_COLUMN].to_numpy(dtype=float)

X_test_smiles = test_df[SMILES_COLUMN].to_numpy()
y_test = test_df[TARGET_COLUMN].to_numpy(dtype=float)

print(f"Train size: {len(X_train_smiles)}")
print(f"Val size: {len(X_val_smiles)}")
print(f"Test size: {len(X_test_smiles)}")


## 3. Embedding Functions


In [ ]:
def make_ecfp_features(smiles_list, radius=ECFP_RADIUS, n_bits=ECFP_N_BITS):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    features = np.zeros((len(smiles_list), n_bits), dtype=np.float32)
    for i, smiles in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        fp = generator.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, features[i])
    return features


_DESCRIPTOR_NAMES = [name for name, _ in Descriptors._descList]
_DESCRIPTOR_FUNCTIONS = [func for _, func in Descriptors._descList]

def make_rdkit_descriptor_features(smiles_list):
    rows = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            rows.append([np.nan] * len(_DESCRIPTOR_FUNCTIONS))
            continue
        values = []
        for func in _DESCRIPTOR_FUNCTIONS:
            try:
                values.append(func(mol))
            except Exception:
                values.append(np.nan)
        rows.append(values)
    return pd.DataFrame(rows, columns=_DESCRIPTOR_NAMES).replace([np.inf, -np.inf], np.nan)


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def _extract_last_hidden_state(outputs):
    if hasattr(outputs, "last_hidden_state") and outputs.last_hidden_state is not None:
        return outputs.last_hidden_state
    if hasattr(outputs, "hidden_states") and outputs.hidden_states is not None:
        return outputs.hidden_states[-1]
    raise ValueError("Model output does not contain hidden states. Try loading with output_hidden_states=True.")


def load_embedding_model(model_name, model_class, trust_remote_code):
    from transformers import AutoModel, AutoModelForMaskedLM

    common_kwargs = {
        "trust_remote_code": trust_remote_code,
        "output_hidden_states": True,
    }

    if model_class == "masked_lm":
        return AutoModelForMaskedLM.from_pretrained(model_name, **common_kwargs)

    if model_class == "roberta_regression":
        try:
            from transformers import RobertaForRegression
            return RobertaForRegression.from_pretrained(model_name, **common_kwargs)
        except ImportError:
            print(
                "RobertaForRegression is unavailable in this transformers install; "
                "falling back to AutoModel for hidden-state embeddings."
            )
            return AutoModel.from_pretrained(model_name, **common_kwargs)

    if model_class == "auto_model":
        return AutoModel.from_pretrained(model_name, **common_kwargs)

    raise ValueError(f"Unknown model_class: {model_class}")


def make_transformer_embeddings(
    smiles_list,
    model_name,
    model_class,
    trust_remote_code=False,
    batch_size=TRANSFORMER_BATCH_SIZE,
    max_length=TRANSFORMER_MAX_LENGTH,
):
    import torch
    from transformers import AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=trust_remote_code,
    )
    model = load_embedding_model(model_name, model_class, trust_remote_code)

    model = model.to(device)
    model.eval()

    embeddings = []
    with torch.no_grad():
        for start in range(0, len(smiles_list), batch_size):
            batch = list(smiles_list[start:start + batch_size])
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            outputs = model(**encoded)
            hidden = _extract_last_hidden_state(outputs)
            pooled = mean_pool(hidden, encoded["attention_mask"])
            embeddings.append(pooled.cpu().numpy())

    return np.vstack(embeddings).astype(np.float32)


In [15]:
embedding_builders = {
    "ECFP": make_ecfp_features,
    "RDKit_descriptors": make_rdkit_descriptor_features,
}

for embedding_name, spec in TRANSFORMER_MODELS.items():
    embedding_builders[embedding_name] = lambda smiles_list, spec=spec: make_transformer_embeddings(
        smiles_list,
        model_name=spec["model_name"],
        model_class=spec["model_class"],
        trust_remote_code=spec["trust_remote_code"],
    )

X_train_by_embedding = {}
X_val_by_embedding = {}
X_test_by_embedding = {}

for embedding_name, builder in embedding_builders.items():
    print(f"Building {embedding_name} features...")
    X_train_by_embedding[embedding_name] = builder(X_train_smiles)
    X_val_by_embedding[embedding_name] = builder(X_val_smiles)
    X_test_by_embedding[embedding_name] = builder(X_test_smiles)
    print(
        f"  train: {X_train_by_embedding[embedding_name].shape}; "
        f"val: {X_val_by_embedding[embedding_name].shape}; "
        f"test: {X_test_by_embedding[embedding_name].shape}"
    )


  train: (1528, 217); val: (191, 217); test: (192, 217)
Building ChemBERTa_77M_MLM features...
  train: (1528, 384); val: (191, 384); test: (192, 384)


## 4. Model Definitions And Hyperparameter Search Spaces

Each model is tuned with 5-fold CV on the training split only. The validation and test splits are not used during cross-validation.


In [16]:
models = {
    "RandomForest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        "params": {
            "model__n_estimators": [300, 600],
            "model__max_depth": [None, 10, 25],
            "model__min_samples_leaf": [1, 3],
        },
    },
    "SVR": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR()),
        ]),
        "params": {
            "model__C": [1, 10, 100],
            "model__epsilon": [0.01, 0.1, 0.2],
            "model__gamma": ["scale", "auto"],
        },
    },
    "Lasso": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Lasso(random_state=RANDOM_STATE, max_iter=20000)),
        ]),
        "params": {
            "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
        },
    },
    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
                eval_metric="rmse",
            )),
        ]),
        "params": {
            "model__n_estimators": [300, 600],
            "model__max_depth": [3, 6],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
        },
    },
}

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


## 5. Train, Tune, And Evaluate


In [17]:
def regression_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": rmse,
        "MAE": mean_absolute_error(y_true, y_pred),
    }


results = []
best_estimators = {}

for embedding_name in X_train_by_embedding:
    X_train = X_train_by_embedding[embedding_name]
    X_val = X_val_by_embedding[embedding_name]
    X_test = X_test_by_embedding[embedding_name]

    for model_name, spec in models.items():
        run_name = f"{embedding_name} + {model_name}"
        print(f"\nTuning {run_name}")

        search = GridSearchCV(
            estimator=spec["pipeline"],
            param_grid=spec["params"],
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            refit=True,
            verbose=1,
        )

        # Hyperparameter optimization is performed only on the training split.
        search.fit(X_train, y_train)

        val_pred = search.predict(X_val)
        test_pred = search.predict(X_test)
        val_metrics = regression_metrics(y_val, val_pred)
        test_metrics = regression_metrics(y_test, test_pred)

        row = {
            "embedding": embedding_name,
            "model": model_name,
            "cv_rmse": -search.best_score_,
            "val_r2": val_metrics["R2"],
            "val_rmse": val_metrics["RMSE"],
            "val_mae": val_metrics["MAE"],
            "test_r2": test_metrics["R2"],
            "test_rmse": test_metrics["RMSE"],
            "test_mae": test_metrics["MAE"],
            "best_params": search.best_params_,
        }
        results.append(row)
        best_estimators[(embedding_name, model_name)] = search.best_estimator_

results_df = pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)
results_df



Tuning ECFP + RandomForest
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Tuning ECFP + SVR
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Tuning ECFP + Lasso
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Tuning ECFP + XGBoost
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Tuning RDKit_descriptors + RandomForest
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Tuning RDKit_descriptors + SVR
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Tuning RDKit_descriptors + Lasso
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Tuning RDKit_descriptors + XGBoost
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Tuning ChemBERTa_77M_MLM + RandomForest
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Tuning ChemBERTa_77M_MLM + SVR
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Tuning ChemBERTa_77M_MLM + Lasso
Fitting 5 folds for each of 5 candidates, totalli

,embedding,model,cv_rmse,val_r2,val_rmse,val_mae,test_r2,test_rmse,test_mae,best_params
0,ChemBERTa_77M_MLM,SVR,0.479531,0.689078,0.466609,0.327806,0.715667,0.495047,0.330335,"{'model__C': 10, 'model__epsilon': 0.01, 'mode..."
1,RDKit_descriptors,XGBoost,0.473035,0.687460,0.467822,0.298755,0.808764,0.405992,0.274484,"{'model__colsample_bytree': 0.8, 'model__learn..."
2,ECFP,XGBoost,0.485933,0.665199,0.484196,0.306981,0.824020,0.389462,0.275516,"{'model__colsample_bytree': 0.8, 'model__learn..."
3,RDKit_descriptors,SVR,0.493221,0.661933,0.486552,0.314657,0.779034,0.436411,0.309597,"{'model__C': 10, 'model__epsilon': 0.1, 'model..."
4,ECFP,RandomForest,0.491581,0.636817,0.504302,0.300304,0.787919,0.427547,0.279297,"{'model__max_depth': None, 'model__min_samples..."
5,RDKit_descriptors,RandomForest,0.526757,0.631362,0.508075,0.321855,0.745413,0.468436,0.297922,"{'model__max_depth': 25, 'model__min_samples_l..."
6,ECFP,SVR,0.506791,0.618928,0.516573,0.311620,0.759400,0.455387,0.302954,"{'model__C': 10, 'model__epsilon': 0.01, 'mode..."
7,ChemBERTa_77M_MLM,XGBoost,0.536302,0.594567,0.532829,0.360192,0.659286,0.541911,0.330227,"{'model__colsample_bytree': 0.8, 'model__learn..."
8,ECFP,Lasso,0.558591,0.538951,0.568200,0.341842,0.772885,0.442441,0.312692,{'model__alpha': 0.01}
9,ChemBERTa_77M_MLM,RandomForest,0.592536,0.523950,0.577370,0.388077,0.596177,0.589968,0.364614,"{'model__max_depth': None, 'model__min_samples..."


In [18]:
pd.set_option("display.max_colwidth", 200)
summary_cols = [
    "embedding",
    "model",
    "cv_rmse",
    "val_rmse",
    "val_mae",
    "val_r2",
    "test_rmse",
    "test_mae",
    "test_r2",
    "best_params",
]
results_df[summary_cols]


,embedding,model,cv_rmse,val_rmse,val_mae,val_r2,test_rmse,test_mae,test_r2,best_params
0,ChemBERTa_77M_MLM,SVR,0.479531,0.466609,0.327806,0.689078,0.495047,0.330335,0.715667,"{'model__C': 10, 'model__epsilon': 0.01, 'model__gamma': 'auto'}"
1,RDKit_descriptors,XGBoost,0.473035,0.467822,0.298755,0.687460,0.405992,0.274484,0.808764,"{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 6, 'model__n_estimators': 600, 'model__subsample': 0.8}"
2,ECFP,XGBoost,0.485933,0.484196,0.306981,0.665199,0.389462,0.275516,0.824020,"{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 6, 'model__n_estimators': 600, 'model__subsample': 0.8}"
3,RDKit_descriptors,SVR,0.493221,0.486552,0.314657,0.661933,0.436411,0.309597,0.779034,"{'model__C': 10, 'model__epsilon': 0.1, 'model__gamma': 'scale'}"
4,ECFP,RandomForest,0.491581,0.504302,0.300304,0.636817,0.427547,0.279297,0.787919,"{'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 300}"
5,RDKit_descriptors,RandomForest,0.526757,0.508075,0.321855,0.631362,0.468436,0.297922,0.745413,"{'model__max_depth': 25, 'model__min_samples_leaf': 1, 'model__n_estimators': 300}"
6,ECFP,SVR,0.506791,0.516573,0.311620,0.618928,0.455387,0.302954,0.759400,"{'model__C': 10, 'model__epsilon': 0.01, 'model__gamma': 'auto'}"
7,ChemBERTa_77M_MLM,XGBoost,0.536302,0.532829,0.360192,0.594567,0.541911,0.330227,0.659286,"{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 6, 'model__n_estimators': 600, 'model__subsample': 0.8}"
8,ECFP,Lasso,0.558591,0.568200,0.341842,0.538951,0.442441,0.312692,0.772885,{'model__alpha': 0.01}
9,ChemBERTa_77M_MLM,RandomForest,0.592536,0.577370,0.388077,0.523950,0.589968,0.364614,0.596177,"{'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 600}"


In [19]:
best_row = results_df.iloc[0]
print("Best model by validation RMSE")
print(best_row[["embedding", "model", "cv_rmse", "val_rmse", "val_mae", "val_r2", "test_rmse", "test_mae", "test_r2"]])
print("Best params:")
print(best_row["best_params"])


Best model by validation RMSE
embedding    ChemBERTa_77M_MLM
model                      SVR
cv_rmse               0.479531
val_rmse              0.466609
val_mae               0.327806
val_r2                0.689078
test_rmse             0.495047
test_mae              0.330335
test_r2               0.715667
Name: 0, dtype: object
Best params:
{'model__C': 10, 'model__epsilon': 0.01, 'model__gamma': 'auto'}


## 6. Save Results


In [20]:
RESULTS_PATH = Path("dpph_embedding_model_results.csv")
results_df.to_csv(RESULTS_PATH, index=False)
print(f"Saved results to {RESULTS_PATH.resolve()}")


Saved results to /Users/zhenjiao-ucd/Downloads/small-molecules/dpph_embedding_model_results.csv


## Notes

- This notebook reads fixed `train.csv`, `val.csv`, and `test.csv` split files.
- `GridSearchCV` uses 5-fold cross-validation only on `train.csv`; validation and test data are evaluated after refitting the best CV model on the full training split.
- Transformer embeddings require the `transformers` and `torch` packages and may need internet access the first time each pretrained model is downloaded.
- DeepChem official loading patterns are used: `ChemBERTa-77M-MTR` uses `RobertaForRegression`, while `ChemBERTa-77M-MLM` and `MoLFormer-c3-1.1B` use `AutoModelForMaskedLM`.
- If `RobertaForRegression` is unavailable in the installed `transformers`, the notebook falls back to `AutoModel` for `ChemBERTa-77M-MTR` hidden-state embeddings.
- If MoLFormer raises a `create_bidirectional_mask` import error, upgrade `transformers` and restart the kernel before re-running the notebook.
- If transformer embedding is slow on CPU, reduce the dataset temporarily or lower `TRANSFORMER_BATCH_SIZE` if memory is limited.
- The selected target is `minusLogIC50 (M)`, so higher predictions correspond to stronger activity under that transformed endpoint.
